In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd "/content/drive/MyDrive/nearest_neighbors/"

/content/drive/MyDrive/nearest_neighbors


In [3]:
import os
import sys
BASE_DIR = os.path.abspath(".")
print(BASE_DIR)
sys.path.append(BASE_DIR)
from collections import Counter
from itertools import chain
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
torch.cuda.empty_cache()
import json
import torch.nn.functional as F
from data_processing.dataloader import Dataloader
from data_processing.dataset import TrainingDataset
from model.query_encoder import QueryEncoder
from model.document_encoder import DocumentEncoder
from model.train import Trainer
%load_ext autoreload
%autoreload 2

/content/drive/MyDrive/nearest_neighbors


In [4]:

dataset_path = os.path.join(BASE_DIR, "dataset_analysis/simulation_output/simulation_results_1753500788.json")
dataloader_instance = Dataloader(batch_size=8, dataset_path=dataset_path, test_size=0.2)
train_dataloader = dataloader_instance.get_train_dataloader()
test_dataloader = dataloader_instance.get_test_dataloader()

for batch in train_dataloader:
    print(len(batch))
    print("Keys in the batch:")
    for key in batch.keys():
        print(key)

    print("Shapes of tensors in the batch:")
    for key, tensor in batch.items():
         print(f"{key}: {tensor.shape}")
    break



for batch in test_dataloader:
    print(len(batch))
    print("Keys in the batch:")
    for key in batch.keys():
        print(key)

    print("Shapes of tensors in the batch:")
    for key, tensor in batch.items():
         print(f"{key}: {tensor.shape}")
    break

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Dataset split into 1584 training and 396 testing samples.
8
Keys in the batch:
query_input_ids
query_attention_mask
positive_document_input_ids
positive_document_attention_mask
positive_numerical_features
negative_document_input_ids
negative_document_attention_mask
negative_numerical_features
Shapes of tensors in the batch:
query_input_ids: torch.Size([8, 128])
query_attention_mask: torch.Size([8, 128])
positive_document_input_ids: torch.Size([8, 128])
positive_document_attention_mask: torch.Size([8, 128])
positive_numerical_features: torch.Size([8, 6])
negative_document_input_ids: torch.Size([8, 128])
negative_document_attention_mask: torch.Size([8, 128])
negative_numerical_features: torch.Size([8, 6])
8
Keys in the batch:
query_input_ids
query_attention_mask
positive_document_input_ids
positive_document_attention_mask
positive_numerical_features
negative_document_input_ids
negative_document_attention_mask
negative_numerical_features
Shapes of tensors in the batch:
query_input_ids: to

In [5]:
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
query_encoder = QueryEncoder(bert_model_name='bert-base-uncased')
document_encoder = DocumentEncoder(
    bert_model_name='bert-base-uncased',
    numerical_dim=6,
    hidden_dim=128
)

trainer = Trainer(
    query_encoder=query_encoder,
    document_encoder=document_encoder,
    device=device,
    temperature=0.07,
    lr=2e-5,
    save_dir="checkpoints"
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [6]:
num_epochs = 5
loss_per_epoch = trainer.train(train_dataloader, num_epochs=num_epochs)
loss_per_epoch = list(enumerate(loss_per_epoch, start=1))
best_epoch, best_loss = min(loss_per_epoch, key=lambda x: x[1])
query_path = f"checkpoints/epoch_{best_epoch}_query_encoder.pt"
doc_path = f"checkpoints/epoch_{best_epoch}_document_encoder.pt"
print(f"\n Best Epoch: {best_epoch} with Loss = {best_loss:.4f}")
print(f"Saved Weights:")
print(f" - Query Encoder: {query_path}")
print(f" - Document Encoder: {doc_path}")

Epoch: 1: 100%|██████████| 198/198 [01:53<00:00,  1.75it/s]


Epoch 1, Average Loss: 0.3003


Epoch: 2: 100%|██████████| 198/198 [02:00<00:00,  1.64it/s]


Epoch 2, Average Loss: 0.3005


Epoch: 3: 100%|██████████| 198/198 [02:00<00:00,  1.65it/s]


Epoch 3, Average Loss: 0.3000


Epoch: 4: 100%|██████████| 198/198 [02:00<00:00,  1.64it/s]


Epoch 4, Average Loss: 0.2999


Epoch: 5: 100%|██████████| 198/198 [02:00<00:00,  1.65it/s]


Epoch 5, Average Loss: 0.2994

 Best Epoch: 5 with Loss = 0.2994
Saved Weights:
 - Query Encoder: checkpoints/epoch_5_query_encoder.pt
 - Document Encoder: checkpoints/epoch_5_document_encoder.pt


In [7]:
trainer.evaluate(test_dataloader)

Evaluating: 100%|██████████| 50/50 [00:12<00:00,  3.99it/s]

Evaluation results:
Average positive similarity: -0.1450
Average negative similarity: -0.1452
Triplet accuracy (pos_sim > neg_sim): 0.5328
Similarity difference statistics:
  Average difference: 0.0002
  Median difference: 0.0000
  Min difference: -0.4398
  Max difference: 0.3892
  Positive differences: 211 (53.28%)
  Zero differences: 0 (0.00%)
  Negative differences: 185 (46.72%)


{'pos_sims': tensor([-0.0972, -0.2169, -0.1780, -0.1774, -0.1912, -0.1686, -0.1931, -0.1499,
         -0.1787, -0.1637, -0.1878, -0.1854, -0.1738, -0.1167,  0.0011, -0.0763,
          0.1857, -0.2024, -0.1905, -0.1802, -0.1850, -0.1296, -0.1463, -0.1971,
         -0.1872, -0.0409, -0.1947, -0.0626, -0.0870, -0.0286, -0.1637, -0.1785,
         -0.1930, -0.1086, -0.1959, -0.1703, -0.1778, -0.1693, -0.1906, -0.2034,
         -0.1789, -0.1834, -0.1191, -0.1920, -0.1726, -0.1549, -0.0559, -0.0520,
         -0.1394, -0.2000,  0.0654, -0.2134, -0.0369, -0.1889, -0.1546, -0.1628,
         -0.0058, -0.1996, -0.1560, -0.1671, -0.2031, -0.1724, -0.0956, -0.1762,
         -0.0874, -0.2012, -0.2012, -0.1494, -0.0986, -0.1900, -0.0099, -0.1351,
         -0.1353, -0.1158, -0.1888, -0.1779, -0.1510, -0.1590, -0.2086, -0.1805,
         -0.1865, -0.0363, -0.1834, -0.1763, -0.0914, -0.1495, -0.1681,  0.0703,
         -0.1974, -0.1967, -0.1954, -0.1995, -0.1885, -0.2107, -0.1757, -0.2018,
         -0.0951